In [11]:
# Se cargan las bibliotecas 
import numpy as np
import time

print("Bibliotecas importadas correctamente")

Bibliotecas importadas correctamente


In [12]:
#Decodifiacion de Hamming (7,4)
#bits_rx es la lista de bits recibida del demodulador con ruido

def hamming_7_4_decode(bits_rx):
    
    bits_rx = np.array(bits_rx, dtype=int)           #lo convierte a un array de ints
    total_blocks = len(bits_rx) // 7

    bits_rx = bits_rx[:total_blocks * 7].reshape((total_blocks, 7))

    decoded = []

    for c in bits_rx:
        # Calculo del síndrome 
        s1 = c[0] ^ c[2] ^ c[4] ^ c[6]  # paridad de las posiciones 1,3,5,7
        s2 = c[1] ^ c[2] ^ c[5] ^ c[6]  # paridad de las posiciones 2,3,6,7
        s3 = c[3] ^ c[4] ^ c[5] ^ c[6]  # paridad de las posiciones 4,5,6,7

        S = s1 + 2 * s2 + 4 * s3    # síndrome 

        if S != 0:         #si el sindrome es = 0 no se detecta error, si sindrome != 0 hay un error en la posicion S-1 (S-1 porque Python inicia en pos 0)
            c[S - 1] ^= 1  # corregir bit 

        # Extraer los bits de datos que estan en las posiciones 3,5,6,7, entonces índices 2,4,5,6 
        #Ignora los bits de paridad
        decoded.extend([c[2], c[4], c[5], c[6]])

    return np.array(decoded, dtype=int)


In [13]:
#Decodifiacion de Hamming (15,11)
#bits_rx es la lista de bits recibida del demodulador con ruido

def hamming_15_11_decode(bits_rx):

    bits_rx = np.array(bits_rx, dtype=int)
    total_blocks = len(bits_rx) // 15

    bits_rx = bits_rx[:total_blocks * 15].reshape((total_blocks, 15))

    decoded = []

    for c in bits_rx:
        # Calculo del síndrome de Hamming(15,11)
        s1 = c[0] ^ c[2] ^ c[4] ^ c[6] ^ c[8] ^ c[10] ^ c[12] ^ c[14]    # paridad de las posiciones 1,3,5,7,9,11,13,15
        s2 = c[1] ^ c[2] ^ c[5] ^ c[6] ^ c[9] ^ c[10] ^ c[13] ^ c[14]    # paridad de las posiciones 2,3,6,7,10,11,14,15
        s4 = c[3] ^ c[4] ^ c[5] ^ c[6] ^ c[11] ^ c[12] ^ c[13] ^ c[14]   # paridad de las posiciones 4,5,6,7,12,13,14,15
        s8 = c[7] ^ c[8] ^ c[9] ^ c[10] ^ c[11] ^ c[12] ^ c[13] ^ c[14]  # paridad de las posiciones 8,9,10,11,12,13,14,15

        S = s1 + 2 * s2 + 4 * s4 + 8 * s8  

        if S != 0:
            c[S - 1] ^= 1  

        # Extraer datos: posiciones 3,5,6,7,9,10,11,12,13,14,15
        data_idx = [2, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14]
        decoded.extend(c[data_idx])

    return np.array(decoded, dtype=int)

In [14]:
#Funcion para pasar de bits a bytes 

def bits_to_bytes(bits):
    
    bits = np.array(bits, dtype=int)
    num_bytes = len(bits) // 8
    bits = bits[:num_bytes*8].reshape((num_bytes, 8))

    values = []
    for byte in bits:
        v = 0
        for b in byte:
            v = (v << 1) | int(b)
        values.append(v)
    return np.array(values, dtype=int)


In [15]:
bits_rx_74 = [1,0,1,1,1,0,1, 0,1,0,1,1,1,0]   # dos codewords de 7 bits con errores
decoded_bits_74 = hamming_7_4_decode(bits_rx_74)
bpm_rec_74 = bits_to_bytes(decoded_bits_74)

print("BPM reconstruidos (Hamming 7,4):", bpm_rec_74)

BPM reconstruidos (Hamming 7,4): [210]


In [16]:
bits_rx_1511 = [1,0,1,1,0,1,1,1,0,1,0,1,1,1,1,
                0,0,1,0,1,1,1,0,0,1,1,0,1,1,1]

decoded_bits_1511 = hamming_15_11_decode(bits_rx_1511)
decoded_bits_1511 = decoded_bits_1511.reshape((-1, 11))

# 2) De cada bloque nos interesan solo los primeros 8 bits (d0 a d7), porque d8,d9,d10 eran ceros de relleno en el codificador
payload_bits = decoded_bits_1511[:, :8].reshape(-1)  

# 3) Convertimos esos bits a bytes / BPM
bpm_rec_1511 = bits_to_bytes(payload_bits)

print("BPM reconstruidos (Hamming 15,11):", bpm_rec_1511)


BPM reconstruidos (Hamming 15,11): [245 242]
